# Bank Account Fraud Detection Model
## Fall 2026
### Model Owner: Joshua Sanchez
### Model Validator: Anna Pope


# Section I: Problem Statement / Importance of the Model
Bank account fraud is a costly problem for financial institutions. Fraudsters open accounts with stolen or synthetic identities, then use them for payment fraud, money mule activity, and credit abuse. According to TransUnion’s recent H2 2025 Update: Top Fraud Trends, 8.3% (or 1 in 12) of all new accounts created in H1 2025 were suspected to be fraudulent (TransUnion, 2025).  Manual review cannot keep pace with application volume, so institutions rely on statistical models to score each application at the time it is submitted.
This model estimates the probability that a bank account opening application is fraudulent, then given a high enough probability, it flags the application as fraudulent. The target is 'fraud_bool', where 1 marks a fraudulent application, and 0 marks a legitimate one. Its output supports a decision to approve, decline, or send the application for additional review.
The model uses features from four groups.
Applicant and identity signals: income, customer_age, name_email_similarity, email_is_free, phone_home_valid, phone_mobile_valid, employment_status, housing_status
Address and banking history: prev_address_months_count, current_address_months_count, bank_months_count, has_other_cards, bank_branch_count_8w
Application and request behavior: payment_type, source, proposed_credit_limit, intended_balcon_amount, foreign_request, days_since_request, zip_count_4w
Velocity and device signals: velocity_6h, velocity_24h, velocity_4w, date_of_birth_distinct_emails_4w, device_os, keep_alive_session, session_length_in_minutes, device_distinct_emails_8w, credit_risk_score
Four columns are excluded from the feature set: fraud_bool (the target), device_fraud_count (Values are only zero), velocity_4h, and velocity_24h (see Section II). Five features are categorical (payment_type, employment_status, housing_status, source, device_os), and the rest are numeric or boolean.

TransUnion. (2025, December 29). What is new account creation fraud and how to combat it. https://www.transunion.com/blog/new-account-creation-fraud-and-how-to-combat-it-update

# Section II: Business Context / Scope of Model
Fraud detection models are systematic approaches to identify, and therefore prevent, fraudulent activity. This model sits at the front of the account opening process. It scores each application so the bank can approve it, decline it, or refer it for review.
The institution this model is built for is assumed to be a large US bank with more than $30 billion in total assets. That is the size where the interagency model risk management guidance, SR 26-2, is expected to be most relevant. Under that guidance, the depth of validation and monitoring should match the model's materiality, so a model that screens a large volume of applications warrants a documented development process, independent validation, and ongoing performance monitoring. 

Intended use: Score individual account opening applications using only information available at the time of application and estimate the probability of fraud.
Not intended for: Card transaction fraud, ACH or wire fraud, insurance or tax fraud, anti-money laundering monitoring, account takeover, or portfolios that differ materially from the development data.
Why logistic regression: It is interpretable and outputs probabilities. Each coefficient converts to an odds ratio, so analysts, validators, and regulators can see how each feature moves fraud risk. That supports the effective challenge that SR 26-2 treats as a central expectation. Because it outputs probabilities, the decision threshold can be adjusted to the bank's tolerance for missed fraud versus false alarms.
Assumptions of Logistic Regression:
1. Binary Outcomes: 'fraud_bool" is a binary outcome, 1 for fraudulent, 0 for not fraudulent
2. Independence of Observations: "There are no relationships between individual instances. Each individual instance was
generated using a CTGAN independently of each other, and each instance represents
features to detect fraud, with no links to other instances" (Jesus et al., 2022)
3. No extreme influential outliers: Outliers were screened with the 1.5 times IQR rule on the training set. The rule flagged large shares of several columns, up to 24 percent for proposed_credit_limit, because of skewed or discrete distributions and negative missing codes, so it was not used to remove records. Outliers in customer_age (5.1 percent fraud rate), credit_risk_score (4.3 percent), device_distinct_emails_8w (3.7 percent), and proposed_credit_limit (2.1 percent) showed fraud rates well above the 1.1 percent base rate, so they were retained as informative. No observations were removed.
4. No severe multicollinearity: The velocity columns (velocity_6h, velocity_24h, velocity_4w) are highly correlated; velocity_24h was chosen for this model, as there were no NaN values and it still well represents the velocity of applications in a small timeframe. 
5. Linearity of the log odds: Each numeric feature should relate linearly to the log odds of fraud. This was checked on the training set with Box-Tidwell tests, which add an x times natural log of x term for each continuous feature and test whether it is significant, using a Bonferroni-adjusted cutoff of 0.05 divided by the number of tests. Nine of 15 continuous features showed a significant departure from linearity: current_address_months_count, date_of_birth_distinct_emails_4w, credit_risk_score, proposed_credit_limit, name_email_similarity, bank_branch_count_8w, income, zip_count_4w, and bank_months_count. Six showed no significant departure: intended_balcon_amount, session_length_in_minutes, prev_address_months_count, velocity_24h, days_since_request, and customer_age. The tests for prev_address_months_count and intended_balcon_amount used fewer fraud cases (620 and 869) and have lower power. Because the large sample makes small departures significant, the test is an indicator and not proof of practical curvature. No capping or transformation was applied, so any curvature in the flagged features is a limitation of the model.

Data source: Bank Account Fraud (BAF) dataset, released with NeurIPS 2022. Source: https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022

Jesus, S., Pombal, J., Alves, D., Cruz, A., Saleiro, P., Ribeiro, R. P., Gama, J., & Bizarro, P. (2022). Turning the tables: Biased, imbalanced, dynamic tabular datasets for ML evaluation. Advances in Neural Information Processing Systems, 35.

# Section III: Data Information
The dataset is synthetic. It was generated from real, anonymized bank account opening applications, with privacy-preserving techniques applied, so it reflects realistic patterns but does not contain real customers.
Time period: The data spans eight months, recorded in the month column. 
Number of rows: 1,000,000. Number of fraud cases: 11,029. Fraud rate: 0.011029. The published dataset has a fraud rate of roughly 1 percent, so the classes are highly imbalanced.
Fraud definition: An application is labeled fraudulent or not fraudulent.
Data cleaning: The dataset has no blank cells, but missing values are not coded as NaN. Several columns use negative values, most often negative one, as a missing value code. Examples include prev_address_months_count, current_address_months_count, bank_months_count, session_length_in_minutes, and device_distinct_emails_8w. Each missing value (negative) was given a NaN, and the model is able to understand that NaN as an indicator, so it does not take away from the model's strength. If they were left as is, document that the scaler and model treat negative one as a real numeric value, which is a known limitation. The device_fraud_count column was excluded because it is constant in the data, so it carries no information. 
Data preprocessing: columns were separated into five categorical columns, which were one-hot encoded with unseen categories ignored, and the remaining numeric columns, which were standardized. Seven binary or count columns (email_is_free, phone_home_valid, phone_mobile_valid, has_other_cards, foreign_request, keep_alive_session, device_distinct_emails_8w) were cast to integers before modeling. The encoder and scaler were fit on the training set only, then applied to the validation and test sets, which prevents leakage.
Class imbalance: handled with class_weight set to balanced in the logistic regression. No resampling such as SMOTE was used. This follows the project lead's guidance and keeps the validation and test sets at their natural fraud rate.
Train, validation, and test split: 70 percent, 15 percent, 15 percent, stratified on fraud_bool so each set keeps the same fraud rate. The split is random with random_state 27695. A 70/15/15 split was chosen for the training/validation/test sets.

# Section IV: Implementation details
Use the model only to score bank account opening applications that resemble the development data, and only for evaluation in this project. Scores are produced at the time of application using only the model inputs listed in Section I. Applications at or above the approved threshold, currently 0.5, are sent to decline or review.
Monitoring: Track PR AUC, the observed fraud rate, and the share of applications flagged on a regular schedule, and fully revalidate after any material change. I propose monthly monitoring and annual revalidation. SR 26-2 does not set a fixed interval. A schedule can be justified by how deeply a model is reviewed relative to its materiality, how often it changes, and how much data exists to test it. Monitoring should also compare the share of missing values in prev_address_months_count, intended_balcon_amount, and bank_months_count against the development data, because these shares drive the missing indicators.
Performance summary:
• Training set: 700000 applications, 7719 fraud cases
• Validation set: 150000 applications,  1654 fraud cases, ROC AUC: 0.8773, PR AUC: 0.1361
• Test set (out of sample): 150000 applications, 1654 fraud cases, ROC AUC: 0.8705, PR AUC: 0.1298
At the 0.5 threshold, report precision, recall, and F1 for the fraud class on the validation set and the test set. Because class_weight is balanced, expect recall to be high and precision to be low, since the weighting pushes the model to flag more applications. Accuracy is not a useful headline metric, because predicting every application as legitimate would score about 98.9 percent. PR AUC and the fraud class recall and precision are the primary measures. A model that guesses at random has a PR AUC equal to the fraud rate, about 0.011, so PR AUC should be read against that baseline.
Threshold: 0.5 is the default and is not tied to business costs. Choose the threshold on the validation set using the bank's cost of a missed fraud versus the cost of reviewing or declining a legitimate applicant. A "goal recall" was set based on the decision threshold, aiming for recall = 0.9. The final threshold should be confirmed on the test set once.

# Section V: Adaptability/Limitations of the model
How the model changes with the data:
Retraining: refit the encoder, scaler, and model on newer data as fraud patterns shift.
New features: add a column to numeric_cols or cat_cols and refit. The model cannot use a feature it was not trained on.
Tunable settings: regularization strength, class weights, and the decision threshold can be retuned on validation data.
Currently excluded:  fraud_bool (the target), device_fraud_count (Values are only zero), velocity_4h, and velocity_24h (see Section II).
Development data range. The model was trained on 700000 applications, of which 7721 (1.103 percent) are fraud, covering eight months.
Income range: Annual income of the applicant, in decile form. Ranges between [0.1, 0.9].
Name email similarity range: Metric of similarity between email and applicant's name. Higher values represent higher similarity. Ranges between [0,1].
Previous address months count range: Number of months in previous registered address of the applicant, if applicable. Ranges between [-1, 80] months (-1 is a missing value).
Current address months count range: Months in the applicant's currently registered address. Ranges between [-1, 429] months (-1 is a missing value).
Customer age range: Applicant's age in years, rounded to the decade. Ranges between [10, 90] years.
Days since request range: Number of days passed since application was done. Ranges between [0, 79] days.
Intended balcon amount range: Initial transferred amount for application. Ranges between [-16, 114] (negatives are missing values).
Payment type: Credit payment plan type. 5 possible anonymized values.
Zip count 4w range: Number of applications within same zip code in last 4 weeks. Ranges between [1, 6830].
Velocity 6h range: Velocity of total applications made in last 6 hours, i.e., average number of applications per hour in the last 6 hours. Ranges between [-175, 16818].
Velocity 24h range: Velocity of total applications made in last 24 hours, i.e., average number of applications per hour in the last 24 hours. Ranges between [1297, 9586].
Velocity 4w range: Velocity of total applications made in last 4 weeks, i.e., average number of applications per hour in the last 4 weeks. Ranges between [2825, 7020].
Bank branch count 8w range: Number of total applications in the selected bank branch in last 8 weeks. Ranges between [0, 2404].
Date of birth distinct emails 4w range: Number of emails for applicants with same date of birth in last 4 weeks. Ranges between [0, 39].
Employment status: Employment status of the applicant. 7 possible anonymized values.
Credit risk score range: Internal score of application risk. Ranges between [-191, 389].
Email is free: Domain of application email (either free or paid).
Housing status: Current residential status for applicant. 7 possible anonymized values.
Phone home valid: Validity of provided home phone.
Phone mobile valid: Validity of provided mobile phone.
Bank months count range: How old is previous account (if held) in months. Ranges between [-1, 32] months (-1 is a missing value).
Has other cards: If applicant has other cards from the same banking company.
Proposed credit limit range: Applicant's proposed credit limit. Ranges between [200, 2000].
Foreign request: If origin country of request is different from bank's country.
Source: Online source of application. Either browser (INTERNET) or app (TELEAPP).
Session length in minutes range: Length of user session in banking website in minutes. Ranges between [-1, 107] minutes (-1 is a missing value).
Device OS: Operating system of the device that made the request. Possible values are Windows, macOS, Linux, X11, or other.
Keep alive session: User option on session logout.
Device distinct emails 8w range: Number of distinct emails in banking website from the used device in last 8 weeks. Ranges between [-1, 2] emails (-1 is a missing value).
Device fraud count range: Number of fraudulent applications with the used device. Ranges between [0, 1].
Month range: Month when the application was made. Ranges between [0, 7].
Fraud bool: If the application is fraudulent or not.
Country: The dataset documentation does not identify the country or institution.
Behavior outside these ranges is not supported by this model, and predictions on applications outside the development ranges should not be relied on.

Known limitations:
• Logistic regression does not capture interactions or nonlinear effects unless they are engineered into the features.
• Class weighting inflates predicted probabilities relative to the true fraud rate, so scores are rankings more than calibrated probabilities. If calibrated probabilities are needed, add a calibration step and check a calibration curve.
• The default 0.5 threshold flags many false alarms for a rare event, with increasing false positive rate as the threshold is decreased (see Section III).
• Missing value codes: Missing values coded as negative numbers (negative one in five columns, and negative values in intended_balcon_amount) were converted to NaN, imputed with the training set median, and paired with missing indicator columns for prev_address_months_count, intended_balcon_amount, and bank_months_count. This keeps a code such as negative one from being treated as a real value, which would distort the coefficients. The limitation is that median imputation understates variance, and the model depends on how the bank records missing data. A change in recording practice would change the indicator columns and the scores.
• Linearity: Box-Tidwell tests found significant departures from linearity in 9 of 15 continuous features, and the model does not correct for them. No capping or transformation was applied, which keeps each coefficient interpretable as a single odds ratio. Splines or binning could improve fit but would reduce that interpretability.
• The data is synthetic, so real-world performance may differ.

Models considered and not selected. 
The main reason for choosing logistic regression was interpretability and scope.
Random Forest: Not chosen because it gives no coefficients to explain a decision, is harder to validate, and can overfit when fraud cases are scarce.
Gradient Boosted Trees: Not chosen because it is sensitive to hyperparameter tuning, is harder to explain, and adds validation workload. It is a strong challenger model for future work and likely to perform better on this kind of tabular data.
Neural Networks: Not chosen because they need more fraud examples to train reliably, are hard to interpret, and carry a high overfitting risk relative to the benefit.

# Section VI: Reproducibility
Code location: [INSERT link and the exact notebook or file name].
Data: file name Base.csv, source: https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022
Random seeds: every place randomness enters uses random_state 27695, which covers both calls to train_test_split.
Split design: 70/15/15, stratified on fraud_bool. The first split sets aside 15 percent for testing, and the second splits the remainder into training and validation using test_size 0.15 divided by 0.85.
Software environment: Python [3.13.15], pandas [2.3.3], numpy [2.5.2], scikit-learn [1.9.0].
Run order:
1. Load the data and cast the seven binary and count columns to integers. Convert the negative missing codes to NaN and create the missing indicator columns for prev_address_months_count, intended_balcon_amount, and bank_months_count. Missing data is not NaN or blank; it is negative in this dataset, but it is important to indicate it, as it could be a sign that the application is fraudulent.
2. Split into training, validation, and test sets.
3. Fit the OneHotEncoder on the training categorical columns only, then transform all three sets.
4. Fit the StandardScaler on the training numeric columns only, then transform all three sets.
5. Combine the numeric and encoded categorical arrays.
6. Fit LogisticRegression with class_weight=balanced and max_iter=10000 on the training set.
7. Evaluate on the validation set, then on the test set once.
Expected outputs: the numbers a correct rerun should produce are listed in the Section III performance summary. 